# OpenDSS Demonstrations

by Frank M Gonzales, PE, MSEE

In [ ]:
from opendss.pf.powerflow import run_pf
from opendss.la.load_allocation import run_load_allocation
from opendss.scd.short_circuit import calc_sc
from opendss.hc.hosting_capacity import run_hosting_capacity
from opendss.tools.dss_converter import mc_net_to_opendss


### Bulk convert Multiconductor to OpenDSS

In [ ]:
from opendss_converter import mc_net_to_opendss

import pickle
import importlib
from pathlib import Path
import pandas as pd
import opendss.tools.dss_converter as odc

importlib.reload(odc)

input_dir = Path("networks")
output_dir = Path("networks")
output_dir.mkdir(parents=True, exist_ok=True)

max_files = None      # e.g. 25 for batch runs; None = all
overwrite = False     # True to regenerate existing .dss files

pkl_files = sorted(input_dir.glob("*.pkl"))
if max_files is not None:
    pkl_files = pkl_files[:max_files]

print(f"Found {len(pkl_files)} pickle files to process from {input_dir}")

converted = []
skipped = []
failed = []

for i, pkl_path in enumerate(pkl_files, start=1):
    dss_path = output_dir / f"{pkl_path.stem}.dss"
    if dss_path.exists() and not overwrite:
        skipped.append((pkl_path.name, dss_path.name, "exists"))
        continue

    try:
        with pkl_path.open("rb") as f:
            mc_net = pickle.load(f)
        odc.mc_net_to_opendss(mc_net, str(dss_path))
        converted.append((pkl_path.name, dss_path.name))
    except Exception as exc:
        failed.append((pkl_path.name, str(exc)))

    if i % 25 == 0 or i == len(pkl_files):
        print(f"Progress: {i}/{len(pkl_files)} | converted={len(converted)} skipped={len(skipped)} failed={len(failed)}")

print(f"Converted: {len(converted)}")
if converted:
    display(pd.DataFrame(converted, columns=["pickle", "dss"]).head(50))

print(f"Skipped: {len(skipped)}")
if skipped:
    display(pd.DataFrame(skipped, columns=["pickle", "dss", "reason"]).head(50))

print(f"Failed: {len(failed)}")
if failed:
    display(pd.DataFrame(failed, columns=["pickle", "error"]).head(50))



## OpenDSS Load Flow

In [ ]:
# Mean Vpu per network (OpenDSS files)
from pathlib import Path
import numpy as np
import pandas as pd
from dss import dss

opendss_dir = Path("")
dss_files = sorted(opendss_dir.glob("*.dss"))
if not dss_files:
    raise ValueError(f"No .dss files found in {opendss_dir.resolve()}")

rows = []
for file_path in dss_files:
    dss.Text.Command = "clear"
    dss.Text.Command = f"redirect {file_path.as_posix()}"
    dss.Text.Command = "solve"

    vpu = np.asarray(dss.ActiveCircuit.AllBusVmagPu, dtype=float)
    finite_mask = np.isfinite(vpu)

    rows.append({
        "network": file_path.stem,
        "converged": bool(dss.ActiveCircuit.Solution.Converged),
        "node_count": int(finite_mask.sum()),
        "vpu_mean": float(np.nanmean(vpu)) if finite_mask.any() else np.nan,
        "vpu_min": float(np.nanmin(vpu)) if finite_mask.any() else np.nan,
        "vpu_max": float(np.nanmax(vpu)) if finite_mask.any() else np.nan,
    })

vpu_by_network_df = pd.DataFrame(rows).sort_values("network").reset_index(drop=True)
num_cols = ["vpu_mean", "vpu_min", "vpu_max"]
vpu_by_network_df[num_cols] = vpu_by_network_df[num_cols].round(6)

print(f"Networks processed: {len(vpu_by_network_df)}")
print("Mean Vpu per network:")
display(vpu_by_network_df[["network", "converged", "node_count", "vpu_mean"]])

print("\nSummary stats (vpu_mean):")
display(vpu_by_network_df["vpu_mean"].describe().to_frame("vpu_mean"))

#### Show results

In [2]:
from opendss.pf.powerflow import run_pf
from opendss.la.load_allocation import run_load_allocation
from opendss.scd.short_circuit import calc_sc
from opendss.hc.hosting_capacity import run_hosting_capacity

dss_file_or_obj ="c:\\Users\\fgonzales\\git\\mc-0.0.1.15\\networks_final_opendss\\CKT_114_16955.dss"

pf_results = run_pf(dss_file_or_obj)
pf_results

{'converged': True,
 'iterations': 2,
 'res_bus':                    bus   kv_base  num_phases     vm_pu   va_degree  \
 0       -8888882187627  9.237604           3  1.000002 -119.999923   
 1             23320992  9.237604           3  1.000002 -119.999923   
 2             23250568  9.237604           3  1.000002 -119.999922   
 3             23322775  9.237604           3  1.000002 -119.999922   
 4             23322700  9.237604           1  1.000077  120.006039   
 ...                ...       ...         ...       ...         ...   
 2263  -999999186001531  0.069282           1  1.000158  120.010170   
 2264   -99999923248871  0.069282           2  1.000062 -119.994764   
 2265   -99999923243340  9.237604           1       NaN    0.000000   
 2266   -99999923299833  0.069282           1  1.000667  120.035540   
 2267   -99999923249283  0.069282           2  1.000062 -119.994764   
 
                                         vm_pu_per_phase  \
 0     [0.999999334137943, 1.00000195

### Load Allocation

In [3]:
run_load_allocation(dss_file_or_obj, target_kw=1000, target_kvar=500, max_iterations=10, tol=1e-3)

  Iter 1: P_sub=-330.7 kW, target=1000.0, mismatch=1330.7 kW, scale=1.000000
  Iter 2: P_sub=-330.7 kW, target=1000.0, mismatch=1330.7 kW, scale=-3.024171
  Iter 3: P_sub=-330.7 kW, target=1000.0, mismatch=1330.7 kW, scale=9.145623
  Iter 4: P_sub=-330.7 kW, target=1000.0, mismatch=1330.7 kW, scale=-27.657978
  Iter 5: P_sub=-330.7 kW, target=1000.0, mismatch=1330.7 kW, scale=83.642488
  Iter 6: P_sub=-330.7 kW, target=1000.0, mismatch=1330.7 kW, scale=-252.949736
  Iter 7: P_sub=-330.7 kW, target=1000.0, mismatch=1330.7 kW, scale=764.961553
  Iter 8: P_sub=-330.7 kW, target=1000.0, mismatch=1330.7 kW, scale=-2313.397216
  Iter 9: P_sub=-330.6 kW, target=1000.0, mismatch=1330.6 kW, scale=6995.910675
  Iter 10: P_sub=-330.8 kW, target=1000.0, mismatch=1330.8 kW, scale=-21158.801906
  Iter 11: P_sub=-330.4 kW, target=1000.0, mismatch=1330.4 kW, scale=63970.283457
  Iter 12: P_sub=-331.5 kW, target=1000.0, mismatch=1331.5 kW, scale=-193618.180976
  Iter 13: P_sub=-328.2 kW, target=1000.0,

{'converged': False,
 'iterations': 15,
 'mismatch_kw': 1404.11124474224,
 'scale_factor': -17077831.88810668,
 'res_bus':                    bus   kv_base  num_phases     vm_pu   va_degree  \
 0       -8888882187627  9.237604           3  1.000002 -119.999908   
 1             23320992  9.237604           3  1.000002 -119.999908   
 2             23250568  9.237604           3  1.000002 -119.999907   
 3             23322775  9.237604           3  1.000002 -119.999907   
 4             23322700  9.237604           1  1.000089  120.007048   
 ...                ...       ...         ...       ...         ...   
 2263  -999999186001531  0.069282           1  1.000179  120.011624   
 2264   -99999923248871  0.069282           2  1.000149 -119.989819   
 2265   -99999923243340  9.237604           1       NaN    0.000000   
 2266   -99999923299833  0.069282           1  1.000693  120.037248   
 2267   -99999923249283  0.069282           2  1.000124 -119.991123   
 
                        

### Short Cicuit Duty

In [7]:
calc_sc(dss_file_or_obj, bus="23250218.1.3", fault="3ph")

{'res_bus_sc':             bus   kv_base fault_type     ikss_ka  \
 0  23250218.1.3  9.237604        3ph  100.366886   
 
                                            i_fault_a  v_fault_pu     z_ohm  \
 0  [100366.88625468888, 49713.330977380145, 49713...    0.000026  0.092038   
 
       r_ohm     x_ohm  
 0  0.009204  0.091578  ,
 'dss': <py_dss_interface.DSS.DSS at 0x20681abc1f0>}

### Hosting Capacity

In [ ]:
ica_results = run_hosting_capacity(dss_file_or_obj)
ica_results

## Controls